# Module 3 — Intent Classifier

**Requirement:** Build a multi-class classifier using traditional ML on
the labeled `intent` column (recommended, since the dataset already
provides gold intent labels) to classify the customer's intent.

**Dataset:** `bitext/Bitext-customer-support-llm-chatbot-training-dataset`
— 26,872 instruction/response pairs across 27 fine-grained intents.

**Category mapping (as specified in the task):**
- `greeting` — greeting, goodbye, gratitude
- `order_status` — track_order, delivery_options, delivery_period
- `order_management` — cancel_order, change_order, place_order
- `billing_and_refunds` — check_invoice, get_refund, payment_issue
- `account_management` — create_account, edit_account, delete_account, switch_account, recover_password
- `complaint` — complaint, review
- `out_of_scope` — any other intent not explicitly listed above


In [9]:
# 1. Import libraries
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import joblib

## 2. Load the dataset

In [10]:
dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = dataset["train"].to_pandas()

print(df.shape)
print(df["intent"].nunique(), "unique intents")
df.head()

(26872, 5)
27 unique intents


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


## 3. Map the 27 fine-grained intents into the 7 routing categories

Any intent not explicitly listed in the task's mapping falls back to
`out_of_scope`, as the task only specifies categories for the intents
listed above.


In [11]:
intent_to_category = {
    # greeting
    "greeting": "greeting",
    "goodbye": "greeting",
    "gratitude": "greeting",

    # order_status
    "track_order": "order_status",
    "delivery_options": "order_status",
    "delivery_period": "order_status",

    # order_management
    "cancel_order": "order_management",
    "change_order": "order_management",
    "place_order": "order_management",

    # billing_and_refunds
    "check_invoice": "billing_and_refunds",
    "get_refund": "billing_and_refunds",
    "payment_issue": "billing_and_refunds",

    # account_management
    "create_account": "account_management",
    "edit_account": "account_management",
    "delete_account": "account_management",
    "switch_account": "account_management",
    "recover_password": "account_management",

    # complaint
    "complaint": "complaint",
    "review": "complaint",
}

df["category"] = df["intent"].map(intent_to_category).fillna("out_of_scope")

print(df["category"].value_counts())

category
out_of_scope           10910
account_management      4987
billing_and_refunds     2996
order_management        2993
order_status            2989
complaint               1997
Name: count, dtype: int64


## 4. Train/test split

In [12]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df["instruction"], df["category"],
    test_size=0.2, random_state=42, stratify=df["category"]
)

## 5. Vectorize the text (TF-IDF)

In [13]:
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

## 6. Train the classifier

In [14]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

## 7. Evaluate on the test set

In [15]:
test_preds = clf.predict(X_test)

print("Test accuracy:", accuracy_score(y_test, test_preds))
print(classification_report(y_test, test_preds))

Test accuracy: 0.9906976744186047
                     precision    recall  f1-score   support

 account_management       0.99      1.00      1.00       998
billing_and_refunds       0.99      0.96      0.98       599
          complaint       1.00      1.00      1.00       399
   order_management       0.99      0.99      0.99       599
       order_status       1.00      0.99      0.99       598
       out_of_scope       0.99      0.99      0.99      2182

           accuracy                           0.99      5375
          macro avg       0.99      0.99      0.99      5375
       weighted avg       0.99      0.99      0.99      5375



## 8. Save the model and vectorizer (for deployment)

In [16]:
joblib.dump(clf, "intent_classifier_model.pkl")
joblib.dump(vectorizer, "intent_classifier_vectorizer.pkl")

['intent_classifier_vectorizer.pkl']

## 9. Cross-validation (checking for overfitting)

To confirm the high test accuracy isn't a fluke of one particular
train/test split, we run 5-fold stratified cross-validation on the full
pipeline (TF-IDF + Logistic Regression) and look at the spread of scores
across folds.


In [17]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline, df["instruction"], df["category"], cv=skf, scoring="accuracy")

print("Cross-validation accuracy per fold:", cv_scores)
print("Mean accuracy:", cv_scores.mean())
print("Standard deviation:", cv_scores.std())

Cross-validation accuracy per fold: [0.992      0.99106977 0.99050986 0.99218459 0.99218459]
Mean accuracy: 0.9915897629412935
Standard deviation: 0.0006800763963959236
